# Pipeline de Análise — DeepLabCut vs. TopScan

## Visão Geral

Este script executa a **análise quantitativa e estatística** das trajetórias animais
capturadas por dois sistemas de rastreamento:

- **TopScan** — sistema de referência (*gold standard*), coordenadas já em milímetros.
- **DeepLabCut (DLC)** — modelo de *pose estimation* ResNet-50, coordenadas em pixels
  convertidas para cm durante o pré-processamento.

O pipeline compara os dois sistemas frame a frame, calcula métricas de concordância
e gera visualizações e vídeos anotados para publicação.

---

## Pré-requisitos

### Dependências Python
```bash
pip install pandas numpy scipy scikit-learn matplotlib seaborn pingouin similaritymeasures opencv-python
```

### Arquivo de entrada
Este script consome a saída do **Pipeline de Pré-Processamento — DeepLabCut vs. TopScan.ipynb**.
Certifique-se de que os arquivos `.txt` processados estejam nas pastas configuradas
antes de executar.

### Estrutura de pastas esperada no Google Drive
```
MyDrive/
└── conteudo/
    └── validacao_topscan/
        ├── TOPSCAN/
        │   └── TXT novo/          ← arquivos RATO_XX_topscan.txt  (saída do [1])
        ├── DEEPLABCUT/
        │   └── trajetoria_txt_dlc/ ← arquivos RATO_XX_dlc.txt      (saída do [1])
        ├── videos_ratinho/         ← vídeos originais .mp4
        └── resultados/
            └── videos/            ← vídeos anotados (gerado automaticamente)
```

> **Atenção:** Ajuste os caminhos na seção `CONFIGURAÇÃO` antes de executar.

---

## Parâmetros de Configuração

| Parâmetro | Padrão | Descrição |
|---|---|---|
| `FPS_VIDEO` | `30` | Taxa de quadros dos vídeos |
| `ARENA_LARGURA_CM` / `_ALTURA_CM` | `60.0` | Dimensões físicas da arena em cm |
| `IGNORAR_FRAMES_INICIAIS` | `150` | Frames descartados no início (≈5 s de habituação) |
| `BODYPART_REFERENCIA` | `'centerbody'` | Ponto corporal usado nas estatísticas |
| `LIMITE_SALTO_DISTANCIA` | `100.0` | Salto máximo permitido entre frames (pixels) |
| `PARAMS_SUAVIZACAO` | `window=11, order=3` | Parâmetros do filtro Savitzky-Golay |
| `PARAMS_FILTRO_DBSCAN` | `eps=0.2, min_samples=10` | Clustering para remoção de outliers |
| `RATOS_PARA_GERAR_VIDEO` | `["RATO_15"]` | Lista de ratos para geração de vídeo |

---

## Fluxo do Pipeline

```
executar_pipeline()
│
├── 1. carregar_dados()
│       Detecta arquivos TopScan e DLC por padrão de nome (regex RATO_XX)
│
├── 2. processar_trajetorias()
│       Para cada rato e bodypart:
│       ├── IQR + DBSCAN → remove outliers
│       ├── Savitzky-Golay → suavização cinemática
│       └── Conversão para cm (DLC pixels → cm via escala da arena)
│
├── 3. plotar_trajetorias_comparativas()
│       Trajetórias sobrepostas coloridas por tempo (TopScan vs DLC)
│
├── 4. plotar_trajetorias_brutas()
│       Trajetórias brutas pré-processamento para inspeção visual
│
├── 5. definir_rois() + verificar_rois()
│       Define regiões de interesse (ROI) circulares para objetos familiares/novos
│
├── 6. gerar_videos()
│       Exporta vídeos .mp4 com trajetória anotada frame a frame (via OpenCV)
│
├── 7. calcular_exploracao_comportamental()
│       Classifica cada frame como Objeto Familiar / Objeto Novo / Chão
│       Gera etogramas (broken bar) e calcula Índice de Discriminação (DI)
│
└── 8. executar_estatistica()
        ├── 4.1  Tabela de métricas: RMSE, MAE, Fréchet, DTW, Pearson
        ├── 4.2  Heatmap espacial do erro posicional
        ├── 4.3  ANOVA por estado comportamental + post-hoc Bonferroni
        ├── 4.4  DI: Shapiro-Wilk + teste t pareado (TopScan vs DLC)
        ├── 4.5  Bland-Altman (concordância do DI)
        ├── 4.6  Cross-correlação temporal dos deslocamentos
        ├── 4.7  Erro por zona (centro vs. periferia)
        ├── 4.8  KDE de velocidade + K-S test (distribuição TopScan vs DLC)
        └── 4.9  Dashboard 4-painéis por animal (trajetória, erro, heatmap, velocidade)
```

---

## Como Executar no Google Colab

1. Monte o Google Drive:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```
2. Instale as dependências (célula de setup — executar uma vez).
3. Ajuste os caminhos na seção `CONFIGURAÇÃO`.
4. Execute `executar_pipeline()` ao final do notebook.

Para rodar apenas uma etapa isolada, chame a função correspondente diretamente
após executar as células de imports, configuração e processamento.

---

#CONEXÃO AO SEU DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

##IMPORTS

In [ ]:
import os
import re

import cv2
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
import similaritymeasures
from scipy.signal import correlate, savgol_filter
from scipy.signal.windows import hann
from scipy.stats import ks_2samp, pearsonr, shapiro, ttest_rel
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

##CONFIGURAÇÕES

In [ ]:
DIRETORIO_TOPSCAN_TXT    = '/conteudo/validacao_topscan/TOPSCAN/TXT novo/'
DIRETORIO_DLC_TXT        = '/conteudo/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/'
DIRETORIO_VIDEOS         = '/conteudo/validacao_topscan/videos_ratinho/'
DIRETORIO_OUTPUT_VIDEOS  = '/conteudo/validacao_topscan/resultados/videos/'

FPS_VIDEO        = 30
ARENA_LARGURA_CM = 60.0
ARENA_ALTURA_CM  = 60.0

# Dimensões em pixels (largura, altura) de cada vídeo
DIMENSOES_VIDEOS = {
    "RATO_02": (310, 240), "RATO_05": (310, 240), "RATO_07": (330, 240),
    "RATO_09": (330, 240), "RATO_10": (330, 240), "RATO_12": (330, 224),
    "RATO_15": (330, 224), "RATO_16": (330, 240), "RATO_19": (330, 240),
    "RATO_42": (330, 224),
}

# Parâmetros de filtragem e suavização
PARAMS_SUAVIZACAO      = {'window_length': 11, 'polyorder': 3}
PARAMS_FILTRO_DBSCAN   = {'eps': 0.2, 'min_samples': 10}
LIMITE_SALTO_DISTANCIA = 100.0

# Comportamento
NOMES_OBJETOS_FAMILIARES = ['OBJF', 'OBJA']
NOMES_OBJETOS_NOVOS      = ['OBJJ', 'OBJC', 'OBJN1']
TOPSCAN_AREA_MAP_VIDEO   = {'OBJF': 'Objeto Familiar', 'OBJJ': 'Objeto Novo', 'Floor': 'Chao'}

# Análise
IGNORAR_FRAMES_INICIAIS = 150
BODYPART_REFERENCIA     = 'centerbody'
BODYPART_LEGENDA        = 'nose'
BODYPARTS_ANALISE       = ['centerbody', 'nose']

# Ratos para geração de vídeo (lista vazia = todos)
RATOS_PARA_GERAR_VIDEO   = ["RATO_15"]
SISTEMAS_PARA_GERAR_VIDEO = ["topscan", "dlc"]

# ROI templates (configurar para o experimento)
ROI_TEMPLATES = {
    "config_A": {
        "Familiar": {'tipo': 'circulo', 'x': 92,  'y': 115, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 185, 'y': 118, 'raio': 17},
    },
    "config_B": {
        "Familiar": {'tipo': 'circulo', 'x': 105, 'y': 128, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 190, 'y': 125, 'raio': 17},
    },
    "config_C": {
        "Familiar": {'tipo': 'circulo', 'x': 98,  'y': 125, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 188, 'y': 120, 'raio': 17},
    },
}

RATO_ROI_CONFIG = {
    "RATO_02": "config_C", "RATO_05": "config_C", "RATO_07": "config_B",
    "RATO_09": "config_A", "RATO_10": "config_B", "RATO_12": "config_A",
    "RATO_15": "config_A", "RATO_16": "config_B", "RATO_19": "config_B",
    "RATO_42": "config_A",
}

norm_tempo_color = mcolors.Normalize(vmin=0, vmax=300)

##ESTILO GLOBAL DE VISUALIZAÇÃO

In [ ]:
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        13,
    'axes.titlesize':   19,
    'axes.titleweight': 'bold',
    'axes.labelsize':   15,
    'xtick.labelsize':  13,
    'ytick.labelsize':  13,
    'legend.fontsize':  12,
    'figure.dpi':       100,
    'savefig.dpi':      150,
    'axes.grid':        True,
    'grid.linestyle':   ':',
    'grid.alpha':       0.5,
})

##FUNÇÕES

###IO

In [ ]:
def carregar_dados_dlc_txt(caminho):
    if not os.path.exists(caminho):
        print(f"Erro: DLC TXT não encontrado em '{caminho}'")
        return None
    try:
        df = pd.read_csv(caminho, sep=' ', header=0)
        if not all(c in df.columns for c in ['Frame', 'X', 'Y']):
            print(f"Erro: colunas esperadas ['Frame','X','Y'] ausentes em {caminho}")
            return None
        return df
    except Exception as e:
        print(f"Erro ao carregar DLC TXT '{caminho}': {e}")
        return None


def carregar_dados_topscan_txt(caminho):
    esperadas = ['FrameNum', 'CenterX(mm)', 'CenterY(mm)', 'Areas']
    if not os.path.exists(caminho):
        print(f"Erro: TopScan TXT não encontrado em '{caminho}'")
        return None
    try:
        df = pd.read_csv(caminho, sep=r'\s+', header=0, skipinitialspace=True)
        mapa = {
            'CenterX': 'CenterX(mm)', 'CenterY': 'CenterY(mm)',
            'Area': 'Areas', 'Frame': 'FrameNum',
        }
        df.rename(columns={k: v for k, v in mapa.items() if k in df.columns and v not in df.columns}, inplace=True)
        if not all(c in df.columns for c in esperadas):
            print(f"Erro: colunas esperadas {esperadas} ausentes em {caminho}. Encontradas: {df.columns.tolist()}")
            return None
        return df
    except Exception as e:
        print(f"Erro ao carregar TopScan TXT '{caminho}': {e}")
        return None


def extrair_id_rato(texto):
    match = re.search(r'R[ _\-]?A?T?O?[ _\-]?(\d+)', texto.upper())
    return f"RATO_{match.group(1).zfill(2)}" if match else None

###PROCESSAMENTO

In [ ]:
def adicionar_coluna_tempo(df, col_frame, fps):
    df = df.copy()
    df['time'] = pd.to_numeric(df[col_frame], errors='coerce') / float(fps) if fps > 0 else np.nan
    return df


def estimar_parametros_filtro(x, y):
    Q1_x, Q3_x = np.percentile(x, [25, 75])
    Q1_y, Q3_y = np.percentile(y, [25, 75])
    dispersao = np.mean([Q3_x - Q1_x, Q3_y - Q1_y])

    if   dispersao > 150: limite_iqr = 0.4
    elif dispersao > 120: limite_iqr = 0.6
    elif dispersao > 90:  limite_iqr = 0.8
    elif dispersao > 60:  limite_iqr = 1.0
    elif dispersao > 30:  limite_iqr = 1.2
    else:                 limite_iqr = 1.5

    distancias = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    distancias = distancias[~np.isnan(distancias)]
    salto_p95 = np.percentile(distancias, 95)
    salto_p99 = np.percentile(distancias, 99)
    limite_salto = max(salto_p95 * 1.5, salto_p99, np.mean(distancias) * 3)

    std_dist  = np.nanstd(distancias)
    media_dist = np.nanmean(distancias)
    eps        = min(0.25, max(0.05, media_dist + std_dist * 0.5))
    min_samples = 5 if len(x) < 200 else (10 if len(x) < 1000 else 15)

    return {
        "limite_iqr": round(limite_iqr, 2),
        "params_dbscan": {"eps": round(eps, 3), "min_samples": min_samples},
        "limite_salto": round(limite_salto, 2),
    }


def _filtrar_trajetoria(dados, col_x, col_y, limite_iqr, params_dbscan, limite_salto, sufixo):
    dados[col_x] = pd.to_numeric(dados[col_x], errors='coerce')
    dados[col_y] = pd.to_numeric(dados[col_y], errors='coerce')
    dados.dropna(subset=[col_x, col_y], inplace=True)
    if dados.empty:
        return None

    Q1_x, Q3_x = dados[col_x].quantile([0.25, 0.75])
    Q1_y, Q3_y = dados[col_y].quantile([0.25, 0.75])
    dados = dados[
        dados[col_x].between(Q1_x - limite_iqr * (Q3_x - Q1_x), Q3_x + limite_iqr * (Q3_x - Q1_x)) &
        dados[col_y].between(Q1_y - limite_iqr * (Q3_y - Q1_y), Q3_y + limite_iqr * (Q3_y - Q1_y))
    ]
    if dados.empty:
        return None

    coords_scaled = StandardScaler().fit_transform(dados[[col_x, col_y]].values)
    labels = DBSCAN(**params_dbscan).fit_predict(coords_scaled)
    dados = dados[labels != -1].copy()
    if dados.empty:
        return None

    distancia = np.sqrt(dados[col_x].diff()**2 + dados[col_y].diff()**2).fillna(0)
    dados = dados[distancia <= limite_salto]

    return dados.rename(columns={col_x: f'x_raw_filtrado{sufixo}', col_y: f'y_raw_filtrado{sufixo}'})


def processar_topscan_filtrado(caminho, limite_iqr, params_dbscan, limite_salto):
    dados = carregar_dados_topscan_txt(caminho)
    if dados is None or dados.empty:
        return None
    return _filtrar_trajetoria(dados, 'CenterX(mm)', 'CenterY(mm)', limite_iqr, params_dbscan, limite_salto, '')


def processar_dlc_filtrado(caminho, limite_iqr, params_dbscan, limite_salto):
    dados = carregar_dados_dlc_txt(caminho)
    if dados is None or dados.empty:
        return None
    return _filtrar_trajetoria(dados, 'X', 'Y', limite_iqr, params_dbscan, limite_salto, '')


def suavizar_com_savgol(df, col_x='x', col_y='y', window_length=11, polyorder=3):
    df = df.copy()
    for col in [col_x, col_y]:
        if col not in df.columns:
            continue
        dados_validos = df[col].dropna()
        w = int(window_length)
        if w % 2 == 0:
            w -= 1
        p = min(int(polyorder), w - 1)
        if len(dados_validos) >= w and len(dados_validos) > p:
            df.loc[dados_validos.index, col] = savgol_filter(dados_validos.values, w, p)
    return df


def determinar_exploracao_dlc(row, definicoes, col_x, col_y):
    if pd.isna(row[col_x]) or pd.isna(row[col_y]):
        return 'Chao'
    ponto = np.array([row[col_x], row[col_y]])
    for nome, info in definicoes.items():
        if info.get('tipo') == 'circulo':
            if np.linalg.norm(ponto - np.array([info['x'], info['y']])) <= info['raio']:
                return f"objeto_{nome}"
    return 'Chao'


def preparar_dados_dlc_para_video(caminho, fps, definicoes_objetos=None):
    dados = carregar_dados_dlc_txt(caminho)
    if dados is None or dados.empty:
        return None
    dados['X'] = pd.to_numeric(dados['X'], errors='coerce')
    dados['Y'] = pd.to_numeric(dados['Y'], errors='coerce')
    df = dados.rename(columns={'Frame': 'FrameNum', 'X': 'x_video_px', 'Y': 'y_video_px'})
    df = adicionar_coluna_tempo(df, 'FrameNum', fps)
    if definicoes_objetos:
        df['exploration_type'] = df.apply(
            determinar_exploracao_dlc, axis=1,
            definicoes=definicoes_objetos, col_x='x_video_px', col_y='y_video_px',
        )
    else:
        df['exploration_type'] = 'N/A'
    return df


def _sincronizar_dados_cm(rato_id, resultados_cm, bodypart=BODYPART_REFERENCIA, ignorar_frames=IGNORAR_FRAMES_INICIAIS):
    """Recupera, sincroniza e filtra frames iniciais. Retorna df_analise ou None."""
    try:
        df_ts  = resultados_cm[rato_id][bodypart]['topscan'].copy()
        df_dlc = resultados_cm[rato_id][bodypart]['dlc'].copy()
    except KeyError:
        return None
    df = pd.merge(df_ts, df_dlc, on='FrameNum', suffixes=('_topscan', '_dlc')).dropna()
    df = df[df['FrameNum'] >= ignorar_frames].copy()
    return df if not df.empty else None

###VISUALIZAÇÃO

In [ ]:
def plotar_trajetoria_temporal_scatter(ax, x_coords, y_coords, time_coords,
                                       cmap_name='viridis', norm_colors=None,
                                       titulo="", label_x="X", label_y="Y",
                                       cor_linha_base='gray', alpha_linha_base=0.5,
                                       tamanho_scatter=10, alpha_scatter=0.7):
    x = np.asarray(x_coords)
    y = np.asarray(y_coords)
    t = np.asarray(time_coords)
    if len(x) == 0:
        ax.set_title(titulo); ax.set_xlabel(label_x); ax.set_ylabel(label_y)
        return None
    valid_t = t[~np.isnan(t)]
    if norm_colors is None and len(valid_t) > 1:
        norm_colors = mcolors.Normalize(vmin=np.min(valid_t), vmax=np.max(valid_t))
    ax.plot(x, y, color=cor_linha_base, alpha=alpha_linha_base)
    ax.scatter(x, y, c=t, cmap=cmap_name, norm=norm_colors, s=tamanho_scatter, alpha=alpha_scatter, zorder=3)
    ax.set_title(titulo); ax.set_xlabel(label_x); ax.set_ylabel(label_y)
    if norm_colors:
        sm = cm.ScalarMappable(cmap=plt.get_cmap(cmap_name), norm=norm_colors)
        sm.set_array([])
        return sm
    return None


def plotar_broken_bar(periodos, nomes, titulo="Períodos de Exploração", tamanho_figura=(10, 4)):
    if len(periodos) != len(nomes):
        print("Erro: número de listas de períodos diferente do número de nomes.")
        return
    fig, ax = plt.subplots(figsize=tamanho_figura)
    cores = plt.colormaps['viridis'].resampled(len(nomes))
    for i, p in enumerate(periodos):
        if p:
            ax.broken_barh(p, (i - 0.4, 0.8), facecolors=cores(i), label=nomes[i])
    ax.set_yticks(range(len(nomes)))
    ax.set_yticklabels(nomes)
    ax.set_xlabel("Frames do Vídeo")
    ax.set_ylabel("Objetos/Zonas")
    ax.set_title(titulo)
    ax.legend()
    plt.tight_layout()
    plt.show()


def _estilo_eixo_trajetoria(ax, titulo, xlim, ylim, fontsize_titulo=19, fontsize_label=15, fontsize_tick=13):
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.tick_params(axis='both', which='major', labelsize=fontsize_tick)
    ax.set_xlabel("X (cm)", fontsize=fontsize_label)
    ax.set_ylabel("Y (cm)", fontsize=fontsize_label)
    ax.set_title(titulo, fontsize=fontsize_titulo, fontweight='bold')

###VÍDEO

In [ ]:
def anotar_video_com_trajetoria_e_legenda(
    caminho_entrada, df_coordenadas, col_x, col_y, col_frame_sync, caminho_saida,
    rois_para_desenhar=None, texto_legenda_fn=None, fps_saida=None,
    cor_trajetoria=(0, 255, 0), cor_ponto=(0, 0, 255), cor_legenda=(255, 255, 255),
    tamanho_fonte=0.4, espessura=1, posicao_legenda=(10, 30), n_pontos_rastro=100,
):
    print(f"Anotando: '{caminho_saida}'")
    if not os.path.exists(caminho_entrada):
        print(f"ERRO: vídeo não encontrado em '{caminho_entrada}'")
        return False

    cap = cv2.VideoCapture(caminho_entrada)
    if not cap.isOpened():
        print(f"ERRO: não foi possível abrir '{caminho_entrada}'")
        return False

    fps_orig = cap.get(cv2.CAP_PROP_FPS)
    fps_saida = fps_saida or (fps_orig if fps_orig > 1 else 30.0)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if largura == 0 or altura == 0:
        print("ERRO: dimensões do vídeo inválidas.")
        cap.release()
        return False

    out = cv2.VideoWriter(caminho_saida, cv2.VideoWriter_fourcc(*'mp4v'), fps_saida, (largura, altura))
    if not out.isOpened():
        print("ERRO: não foi possível criar o vídeo de saída.")
        cap.release()
        return False

    cores_rois = {'Familiar': (255, 100, 100), 'Novo': (100, 100, 255)}
    frame_atual = 0
    rastro = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if rois_para_desenhar:
            for nome, info in rois_para_desenhar.items():
                if info.get('tipo') == 'circulo':
                    cor = cores_rois.get(nome, (0, 255, 255))
                    cv2.circle(frame, (info['x'], info['y']), info['raio'], cor, 2)
                    cv2.putText(frame, nome, (info['x'] - info['raio'], info['y'] - info['raio'] - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, cor, 2)

        linha = df_coordenadas[df_coordenadas[col_frame_sync] == frame_atual]
        if not linha.empty:
            xv = linha[col_x].iloc[0]
            yv = linha[col_y].iloc[0]
            if not (pd.isna(xv) or pd.isna(yv)):
                pt = (int(xv), int(yv))
                rastro.append(pt)
                cv2.circle(frame, pt, 5, cor_ponto, -1)

        pontos = rastro[-n_pontos_rastro:] if isinstance(n_pontos_rastro, int) else rastro
        for i in range(1, len(pontos)):
            if pontos[i - 1] and pontos[i]:
                cv2.line(frame, pontos[i - 1], pontos[i], cor_trajetoria, 2)

        if texto_legenda_fn:
            dados_frame = linha.iloc[0] if not linha.empty else None
            texto = texto_legenda_fn(dados_frame, frame_atual)
            if texto:
                cv2.putText(frame, texto, posicao_legenda, cv2.FONT_HERSHEY_SIMPLEX,
                            tamanho_fonte, cor_legenda, espessura, cv2.LINE_AA)

        out.write(frame)
        frame_atual += 1

    cap.release()
    out.release()
    print(f"  Video salvo em: {caminho_saida}")
    return True

##ETAPA 0: CARREGAR DADOS (Detecta arquivos disponíveis)

In [ ]:
def carregar_dados():
    arquivos_dlc     = {bp: {} for bp in BODYPARTS_ANALISE}
    arquivos_topscan = {}

    for arq in os.listdir(DIRETORIO_DLC_TXT):
        for bp in BODYPARTS_ANALISE:
            if bp in arq.lower():
                rid = extrair_id_rato(arq)
                if rid:
                    arquivos_dlc[bp][rid] = arq

    for arq in os.listdir(DIRETORIO_TOPSCAN_TXT):
        if arq.lower().endswith('.txt'):
            rid = extrair_id_rato(arq)
            if rid:
                num = rid.split("_")[1]
                arquivos_topscan[rid] = {
                    "arquivo_txt": arq,
                    "video_cortado_nome": f"experimento{int(num)}.mp4",
                }

    ratos_dlc     = set(arquivos_dlc[BODYPART_REFERENCIA].keys()) | set(arquivos_dlc.get(BODYPART_LEGENDA, {}).keys())
    ratos_topscan = set(arquivos_topscan.keys())
    ratos_comuns  = ratos_dlc & ratos_topscan

    arquivos_topscan = {r: arquivos_topscan[r] for r in ratos_comuns}
    arquivos_dlc     = {bp: {r: arq for r, arq in arqs.items() if r in ratos_comuns}
                        for bp, arqs in arquivos_dlc.items()}

    print(f"Ratos válidos ({len(ratos_comuns)}): {sorted(ratos_comuns)}")
    return ratos_comuns, arquivos_topscan, arquivos_dlc

##ETAPA 1: PROCESSAR TRAJETÓRIAS

###ETAPA 1.1: CONVERSÃO PARA CMs

In [ ]:
def processar_trajetorias(ratos_comuns, arquivos_topscan, arquivos_dlc):
    """
    Filtra, alinha espacialmente e converte para centímetros.
    Retorna resultados_processados_cm[rato_id][bodypart] = {'topscan': df, 'dlc': df}
    """
    resultados = {}

    for rato_id in sorted(ratos_comuns):
        print(f"\nProcessando {rato_id}...")
        resultados[rato_id] = {}

        for bodypart in BODYPARTS_ANALISE:
            if rato_id not in arquivos_dlc.get(bodypart, {}):
                continue

            caminho_ts  = os.path.join(DIRETORIO_TOPSCAN_TXT, arquivos_topscan[rato_id]["arquivo_txt"])
            caminho_dlc = os.path.join(DIRETORIO_DLC_TXT, arquivos_dlc[bodypart][rato_id])

            dados_ts_brutos  = carregar_dados_topscan_txt(caminho_ts)
            dados_dlc_brutos = carregar_dados_dlc_txt(caminho_dlc)
            if dados_ts_brutos is None or dados_dlc_brutos is None:
                continue

            filtros_ts  = estimar_parametros_filtro(dados_ts_brutos['CenterX(mm)'].dropna().values,
                                                    dados_ts_brutos['CenterY(mm)'].dropna().values)
            filtros_dlc = estimar_parametros_filtro(dados_dlc_brutos['X'].dropna().values,
                                                    dados_dlc_brutos['Y'].dropna().values)

            df_ts_filt  = processar_topscan_filtrado(caminho_ts,  **filtros_ts)
            df_dlc_filt = processar_dlc_filtrado(caminho_dlc, **filtros_dlc)
            if df_ts_filt is None or df_dlc_filt is None:
                continue

            df_dlc_filt.rename(columns={'Frame': 'FrameNum'}, inplace=True, errors='ignore')

            # Alinhamento espacial (TopScan mm -> pixels DLC)
            df_sync = pd.merge(df_ts_filt, df_dlc_filt, on='FrameNum', suffixes=('_ts', '_dlc')).dropna()
            if df_sync.empty:
                continue

            m_x, c_x = np.polyfit(df_sync['x_raw_filtrado_ts'], df_sync['x_raw_filtrado_dlc'], 1)
            m_y, c_y = np.polyfit(df_sync['y_raw_filtrado_ts'], df_sync['y_raw_filtrado_dlc'], 1)

            df_ts_filt['x_aligned_px'] = df_ts_filt['x_raw_filtrado'] * m_x + c_x
            df_ts_filt['y_aligned_px'] = df_ts_filt['y_raw_filtrado'] * m_y + c_y

            # Escala unificada (cm/pixel)
            all_x = pd.concat([df_dlc_filt['x_raw_filtrado'], df_ts_filt['x_aligned_px']]).dropna()
            all_y = pd.concat([df_dlc_filt['y_raw_filtrado'], df_ts_filt['y_aligned_px']]).dropna()
            range_px = max(all_x.max() - all_x.min(), all_y.max() - all_y.min())
            escala   = ARENA_LARGURA_CM / float(range_px) if range_px > 0 else 0
            gmin_x, gmin_y = all_x.min(), all_y.min()

            # Converter para cm
            df_dlc_cm = df_dlc_filt.copy()
            df_dlc_cm['x_cm'] = (df_dlc_cm['x_raw_filtrado'] - gmin_x) * escala
            df_dlc_cm['y_cm'] = (df_dlc_cm['y_raw_filtrado'] - gmin_y) * escala
            df_dlc_cm['x_px'] = df_dlc_cm['x_raw_filtrado']
            df_dlc_cm['y_px'] = df_dlc_cm['y_raw_filtrado']

            df_ts_cm = df_ts_filt.copy()
            df_ts_cm['x_cm'] = (df_ts_cm['x_aligned_px'] - gmin_x) * escala
            df_ts_cm['y_cm'] = (df_ts_cm['y_aligned_px'] - gmin_y) * escala
            df_ts_cm['x_px'] = df_ts_cm['x_aligned_px']
            df_ts_cm['y_px'] = df_ts_cm['y_aligned_px']

            df_dlc_final = suavizar_com_savgol(df_dlc_cm, 'x_cm', 'y_cm', **PARAMS_SUAVIZACAO)
            df_ts_final  = suavizar_com_savgol(df_ts_cm,  'x_cm', 'y_cm', **PARAMS_SUAVIZACAO)
            df_dlc_final = adicionar_coluna_tempo(df_dlc_final, 'FrameNum', FPS_VIDEO)
            df_ts_final  = adicionar_coluna_tempo(df_ts_final,  'FrameNum', FPS_VIDEO)

            resultados[rato_id][bodypart] = {'topscan': df_ts_final, 'dlc': df_dlc_final}
            print(f"  {rato_id} ({bodypart}): escala={escala:.4f} cm/px")

    print("\nProcessamento concluído.")
    return resultados

###ETAPA 1.2: PLOTS COMPARATIVOS (Filtrados, em cm)

In [ ]:
def plotar_trajetorias_comparativas(resultados_cm):
    print("\n--- Trajetórias Comparativas (cm) ---")
    for rato_id, dados_bp in resultados_cm.items():
        if BODYPART_REFERENCIA not in dados_bp:
            continue
        dados   = dados_bp[BODYPART_REFERENCIA]
        df_ts   = dados.get('topscan')
        df_dlc  = dados.get('dlc')

        fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
        fig.suptitle(f"Comparação de Trajetória — {rato_id}", fontsize=22)

        for ax, df, cmap, titulo in [
            (axes[0], df_ts,  'Reds',    "TopScan"),
            (axes[1], df_dlc, 'Purples', "DeepLabCut"),
        ]:
            if df is not None and not df.empty:
                sm = plotar_trajetoria_temporal_scatter(
                    ax, df['x_cm'], df['y_cm'], df.get('time', np.zeros(len(df))),
                    cmap_name=cmap, norm_colors=norm_tempo_color,
                    label_x="X (cm)", label_y="Y (cm)",
                )
                if sm:
                    cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
                    cbar.set_label("Tempo (s)", fontsize=14)
                    cbar.ax.tick_params(labelsize=12)
            _estilo_eixo_trajetoria(ax, titulo, (0, ARENA_LARGURA_CM), (0, ARENA_ALTURA_CM))

        plt.show()

###ETAPA 1.3: PLOTS DE TRAJETÓRIAS BRUTAS

In [ ]:
def plotar_trajetorias_brutas(ratos_comuns, arquivos_topscan, arquivos_dlc):
    print("\n--- Trajetórias Brutas (sem filtro) ---")
    for rato_id in sorted(ratos_comuns):
        caminho_ts  = os.path.join(DIRETORIO_TOPSCAN_TXT, arquivos_topscan[rato_id]["arquivo_txt"])
        caminho_dlc = os.path.join(DIRETORIO_DLC_TXT, arquivos_dlc[BODYPART_REFERENCIA].get(rato_id, ''))

        df_ts  = carregar_dados_topscan_txt(caminho_ts)
        df_dlc = carregar_dados_dlc_txt(caminho_dlc)
        if df_ts is None or df_dlc is None:
            continue

        col_frame_ts = next((c for c in ['FrameNum', 'Frame'] if c in df_ts.columns), df_ts.columns[0])
        df_ts  = adicionar_coluna_tempo(df_ts,  col_frame_ts, FPS_VIDEO)
        df_dlc = adicionar_coluna_tempo(df_dlc, 'Frame',      FPS_VIDEO)

        fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
        fig.suptitle(f"Trajetórias Brutas — {rato_id}", fontsize=22)

        df_plot_ts = df_ts.dropna(subset=['CenterX(mm)', 'CenterY(mm)'])
        if not df_plot_ts.empty:
            plotar_trajetoria_temporal_scatter(axes[0],
                df_plot_ts['CenterX(mm)'], df_plot_ts['CenterY(mm)'], df_plot_ts['time'],
                cmap_name='Reds', label_x="X (mm)", label_y="Y (mm)")
            xmin, xmax = df_plot_ts['CenterX(mm)'].min(), df_plot_ts['CenterX(mm)'].max()
            ymin, ymax = df_plot_ts['CenterY(mm)'].min(), df_plot_ts['CenterY(mm)'].max()
            axes[0].set_xlim(xmin - 5, xmax + 5)
            axes[0].set_ylim(ymin - 5, ymax + 5)
            axes[0].set_box_aspect(480 / 720)

        axes[0].set_title("TopScan", fontsize=19, fontweight='bold')
        axes[0].set_xlabel("X (mm)", fontsize=15)
        axes[0].set_ylabel("Y (mm)", fontsize=15)

        df_plot_dlc = df_dlc.dropna(subset=['X', 'Y'])
        if not df_plot_dlc.empty:
            plotar_trajetoria_temporal_scatter(axes[1],
                df_plot_dlc['X'], df_plot_dlc['Y'], df_plot_dlc['time'],
                cmap_name='Purples', label_x="X (pixels)", label_y="Y (pixels)")
            xmin, xmax = df_plot_dlc['X'].min(), df_plot_dlc['X'].max()
            ymin, ymax = df_plot_dlc['Y'].min(), df_plot_dlc['Y'].max()
            axes[1].set_xlim(xmin - 5, xmax + 5)
            axes[1].set_ylim(ymin - 5, ymax + 5)
            if rato_id in DIMENSOES_VIDEOS:
                larg, alt = DIMENSOES_VIDEOS[rato_id]
                axes[1].set_box_aspect(alt / larg)

        axes[1].set_title("DeepLabCut", fontsize=19, fontweight='bold')
        axes[1].set_xlabel("X (pixels)", fontsize=15)
        axes[1].set_ylabel("Y (pixels)", fontsize=15)

        plt.show()

##ETAPA 2: VÍDEOS (ROIs, preparação e geração)

In [ ]:
def definir_rois(ratos_comuns):
    rois = {}
    for rato_id, config_nome in RATO_ROI_CONFIG.items():
        if rato_id not in ratos_comuns:
            continue
        if config_nome in ROI_TEMPLATES:
            rois[rato_id] = ROI_TEMPLATES[config_nome]
            print(f"  {rato_id} -> {config_nome}")
        else:
            print(f"ERRO: configuração '{config_nome}' não existe em ROI_TEMPLATES.")
    return rois


def verificar_rois(rois_definidas, arquivos_topscan):
    print("\n--- Verificação visual das ROIs ---")
    import random
    for rato_id, rois in rois_definidas.items():
        info = arquivos_topscan.get(rato_id)
        if not info:
            continue
        caminho_video = os.path.join(DIRETORIO_VIDEOS, info["video_cortado_nome"])
        if not os.path.exists(caminho_video):
            print(f"  ERRO: vídeo não encontrado em '{caminho_video}'")
            continue

        cap = cv2.VideoCapture(caminho_video)
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(int(total * 0.1), int(total * 0.9)))
        ret, frame = cap.read()
        cap.release()
        if not ret:
            continue

        for nome, info_roi in rois.items():
            if info_roi.get('tipo') == 'circulo':
                cor = (255, 100, 100) if "Familiar" in nome else (100, 100, 255)
                cv2.circle(frame, (info_roi['x'], info_roi['y']), info_roi['raio'], cor, 2)
                cv2.putText(frame, nome, (info_roi['x'] + info_roi['raio'] + 5, info_roi['y'] + 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"ROI — {rato_id}")
        plt.axis('off')
        plt.show()


def _legenda_exploracao(dados_frame, num_frame):
    if dados_frame is not None and 'exploration_type' in dados_frame and pd.notna(dados_frame['exploration_type']):
        return f"Frame: {num_frame} | {dados_frame['exploration_type'].replace('objeto_', '')}"
    return f"Frame: {num_frame}"


def gerar_videos(ratos_comuns, arquivos_topscan, arquivos_dlc, resultados_cm, rois_definidas):
    os.makedirs(DIRETORIO_OUTPUT_VIDEOS, exist_ok=True)
    ratos = RATOS_PARA_GERAR_VIDEO if RATOS_PARA_GERAR_VIDEO else sorted(ratos_comuns)

    for rato_id in ratos:
        if rato_id not in ratos_comuns:
            print(f"Aviso: '{rato_id}' não está na lista de ratos comuns.")
            continue

        info_ts = arquivos_topscan.get(rato_id)
        rois    = rois_definidas.get(rato_id)
        if not info_ts or not rois:
            print(f"  ERRO: dados ausentes para '{rato_id}'.")
            continue

        caminho_video = os.path.join(DIRETORIO_VIDEOS, info_ts["video_cortado_nome"])
        if not os.path.exists(caminho_video):
            print(f"  ERRO: vídeo não encontrado em '{caminho_video}'.")
            continue

        try:
            df_ts_proc  = resultados_cm[rato_id][BODYPART_REFERENCIA]['topscan'].copy()
            df_dlc_proc = resultados_cm[rato_id][BODYPART_REFERENCIA]['dlc'].copy()
        except KeyError:
            print(f"  Dados processados ausentes para {rato_id}. Pulando.")
            continue

        df_ts_video  = df_ts_proc.rename(columns={"x_px": "x_video_px", "y_px": "y_video_px"})
        df_dlc_video = df_dlc_proc.rename(columns={"x_px": "x_video_px", "y_px": "y_video_px"})

        if 'topscan' in SISTEMAS_PARA_GERAR_VIDEO:
            df_bruto_ts = carregar_dados_topscan_txt(os.path.join(DIRETORIO_TOPSCAN_TXT, info_ts["arquivo_txt"]))
            if df_bruto_ts is not None and 'Areas' in df_bruto_ts.columns:
                df_bruto_ts['exploration_type'] = df_bruto_ts['Areas'].map(TOPSCAN_AREA_MAP_VIDEO).fillna('Chao')
                df_ts_video = pd.merge(df_ts_video[['FrameNum', 'x_video_px', 'y_video_px', 'time']],
                                       df_bruto_ts[['FrameNum', 'exploration_type']], on='FrameNum', how='left')
            nome_saida = os.path.join(DIRETORIO_OUTPUT_VIDEOS,
                                      f"{os.path.splitext(info_ts['video_cortado_nome'])[0]}_TopScan.mp4")
            anotar_video_com_trajetoria_e_legenda(
                caminho_video, df_ts_video, 'x_video_px', 'y_video_px', 'FrameNum', nome_saida,
                rois_para_desenhar=rois, texto_legenda_fn=_legenda_exploracao,
            )

        if 'dlc' in SISTEMAS_PARA_GERAR_VIDEO:
            info_dlc = arquivos_dlc.get(BODYPART_LEGENDA, {}).get(rato_id)
            if info_dlc:
                df_status = preparar_dados_dlc_para_video(
                    os.path.join(DIRETORIO_DLC_TXT, info_dlc), FPS_VIDEO, definicoes_objetos=rois)
                if df_status is not None:
                    df_dlc_video = pd.merge(df_dlc_video[['FrameNum', 'x_video_px', 'y_video_px', 'time']],
                                            df_status[['FrameNum', 'exploration_type']], on='FrameNum', how='left')
            nome_saida = os.path.join(DIRETORIO_OUTPUT_VIDEOS,
                                      f"{os.path.splitext(info_ts['video_cortado_nome'])[0]}_DLC.mp4")
            anotar_video_com_trajetoria_e_legenda(
                caminho_video, df_dlc_video, 'x_video_px', 'y_video_px', 'FrameNum', nome_saida,
                rois_para_desenhar=rois, texto_legenda_fn=_legenda_exploracao,
            )

##ETAPA 3: EXPLORAÇÃO COMPORTAMENTAL (Broken Bar)

In [ ]:
def _gerar_periodos(df_exp):
    if df_exp.empty:
        return []
    periodos = []
    start, ultimo = None, None
    df_exp = df_exp.sort_values('time').dropna(subset=['time'])
    for _, row in df_exp.iterrows():
        t, tipo = row['time'], row['exploration_type']
        if tipo != ultimo:
            if ultimo is not None and start is not None:
                dur = max(0, t - start - 1 / FPS_VIDEO)
                if dur > 1e-9:
                    periodos.append((start, dur, ultimo))
            start, ultimo = t, tipo
    if ultimo is not None and start is not None and not df_exp.empty:
        dur = max(0, df_exp['time'].max() - start - 1 / FPS_VIDEO)
        if dur > 1e-9:
            periodos.append((start, dur, ultimo))
    return periodos


def calcular_exploracao_comportamental(ratos_comuns, arquivos_topscan, arquivos_dlc, rois_definidas):
    tempos_salvos = {}
    cores = {'objeto_Familiar': '#cc5500', 'objeto_Novo': '#5472cc', 'chao': '#E0E0E0'}

    for rato_id in ratos_comuns:
        info_ts  = arquivos_topscan.get(rato_id)
        info_dlc = arquivos_dlc.get(BODYPART_LEGENDA, {}).get(rato_id)
        rois     = rois_definidas.get(rato_id)
        if not all([info_ts, info_dlc, rois]):
            continue

        # --- TopScan ---
        df_ts = carregar_dados_topscan_txt(os.path.join(DIRETORIO_TOPSCAN_TXT, info_ts['arquivo_txt']))
        if df_ts is None or 'Areas' not in df_ts.columns:
            continue
        df_ts = adicionar_coluna_tempo(df_ts, 'FrameNum', FPS_VIDEO)
        df_ts['exploration_type'] = df_ts['Areas'].apply(
            lambda a: 'objeto_Familiar' if a in NOMES_OBJETOS_FAMILIARES
                      else ('objeto_Novo' if a in NOMES_OBJETOS_NOVOS else 'chao')
        )

        # --- DLC ---
        df_dlc = carregar_dados_dlc_txt(os.path.join(DIRETORIO_DLC_TXT, info_dlc))
        if df_dlc is None:
            continue
        df_dlc = adicionar_coluna_tempo(df_dlc, 'Frame', FPS_VIDEO)
        df_dlc['exploration_type'] = df_dlc.apply(
            determinar_exploracao_dlc, axis=1, definicoes=rois, col_x='X', col_y='Y')

        periodos_ts  = _gerar_periodos(df_ts)
        periodos_dlc = _gerar_periodos(df_dlc)

        # Calcular tempos
        tempos = {s: {'objeto_Familiar': 0, 'objeto_Novo': 0}
                  for s in ['topscan', 'dlc']}
        for sistema, periodos in [('topscan', periodos_ts), ('dlc', periodos_dlc)]:
            for _, dur, tipo in periodos:
                if tipo in tempos[sistema]:
                    tempos[sistema][tipo] += dur

        tempos_salvos[rato_id] = {
            "topscan_novo": tempos['topscan']['objeto_Novo'],
            "topscan_familiar": tempos['topscan']['objeto_Familiar'],
            "dlc_novo": tempos['dlc']['objeto_Novo'],
            "dlc_familiar": tempos['dlc']['objeto_Familiar'],
        }

        # --- Broken Bar ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 5), sharex=True, constrained_layout=True)
        for ax, periodos in [(ax1, periodos_ts), (ax2, periodos_dlc)]:
            for start, dur, tipo in periodos:
                ax.broken_barh([(start, dur)], (0, 1), facecolors=cores.get(tipo, '#808080'))

        max_time = max(df_ts['time'].max() if not df_ts.empty else 0,
                       df_dlc['time'].max() if not df_dlc.empty else 0)
        for ax, nome in [(ax1, 'TopScan'), (ax2, 'DeepLabCut')]:
            ax.set_ylim(0, 1); ax.set_yticks([0.5]); ax.set_yticklabels([nome], fontsize=15)
            ax.grid(True, axis='x', linestyle='--', alpha=0.7)
            ax.set_xlim(0, max_time if max_time > 0 else 300)

        ax1.set_title(f' ')
        ax2.set_xlabel('Tempo (segundos)', fontsize=15)
        patches = [mpatches.Patch(color=cores[k], label=k.replace('objeto_', 'Objeto ').capitalize())
                   for k in cores]
        fig.legend(handles=patches, loc='upper right', ncol=len(patches))
        plt.suptitle(f"Exploração Comportamental — {rato_id}", fontsize=16)
        plt.show()

    return tempos_salvos

##ETAPA 4: ESTATÍSTICA

###ETAPA 4.1: Métricas de similaridade de trajetória (cm e pixels)

In [ ]:
def executar_estatistica(ratos_comuns, resultados_cm, arquivos_dlc, rois_definidas, tempos_salvos):
    print("\n=== 4.1 Métricas de Similaridade ===")
    resultados_metricas = []

    for rato_id in sorted(ratos_comuns):
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue

        traj_ts  = df_an[['x_cm_topscan', 'y_cm_topscan']].values
        traj_dlc = df_an[['x_cm_dlc', 'y_cm_dlc']].values

        df_an['distancia_cm'] = np.sqrt(
            (df_an['x_cm_dlc'] - df_an['x_cm_topscan'])**2 +
            (df_an['y_cm_dlc'] - df_an['y_cm_topscan'])**2
        )

        rmse    = np.sqrt(np.mean(df_an['distancia_cm']**2))
        mae     = df_an['distancia_cm'].mean()
        std_err = df_an['distancia_cm'].std()
        frechet = similaritymeasures.frechet_dist(traj_dlc, traj_ts)
        dtw, _  = similaritymeasures.dtw(traj_dlc, traj_ts)
        corr_x, _ = pearsonr(df_an['x_cm_dlc'], df_an['x_cm_topscan'])
        corr_y, _ = pearsonr(df_an['y_cm_dlc'], df_an['y_cm_topscan'])

        resultados_metricas.append({
            "ID Animal": rato_id, "RMSE (cm)": rmse, "MAE (cm)": mae,
            "Std Erro (cm)": std_err, "Fréchet (cm)": frechet,
            "DTW (cm)": dtw, "Corr X": corr_x, "Corr Y": corr_y,
        })

    if resultados_metricas:
        df_met = pd.DataFrame(resultados_metricas)
        pd.set_option('display.float_format', '{:.3f}'.format)
        print(df_met.to_string(index=False))

###ETAPA 4.2: Mapa de calor do erro (cm)

In [ ]:
print("\n=== 4.2 Mapas de Calor do Erro ===")
    NUM_BINS = 50
    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue
        df_an['distancia_cm'] = np.sqrt(
            (df_an['x_cm_dlc'] - df_an['x_cm_topscan'])**2 +
            (df_an['y_cm_dlc'] - df_an['y_cm_topscan'])**2
        )
        bins_x = np.linspace(0, ARENA_LARGURA_CM, NUM_BINS + 1)
        bins_y = np.linspace(0, ARENA_ALTURA_CM,  NUM_BINS + 1)
        hm_sum, _, _ = np.histogram2d(df_an['x_cm_dlc'], df_an['y_cm_dlc'],
                                       bins=[bins_x, bins_y], weights=df_an['distancia_cm'])
        hm_cnt, _, _ = np.histogram2d(df_an['x_cm_dlc'], df_an['y_cm_dlc'], bins=[bins_x, bins_y])
        hm_avg = np.divide(hm_sum, hm_cnt, out=np.zeros_like(hm_sum), where=hm_cnt != 0)

        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(hm_avg.T, origin='lower', cmap='jet',
                       extent=[0, ARENA_LARGURA_CM, 0, ARENA_ALTURA_CM])
        ax.plot(df_an['x_cm_dlc'], df_an['y_cm_dlc'], color='white', alpha=0.3, linewidth=1)
        cbar = fig.colorbar(im, ax=ax); cbar.set_label('Erro Médio (cm)')
        ax.set_title(f'Mapa de Calor do Erro — {rato_id}')
        ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)')
        plt.show()

###ETAPA 4.3: Erro por estado comportamental (ANOVA de medidas repetidas)

In [ ]:
print("\n=== 4.3 Erro por Estado Comportamental ===")
    dados_anova = []
    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue
        df_an['distancia_cm'] = np.sqrt(
            (df_an['x_cm_dlc'] - df_an['x_cm_topscan'])**2 +
            (df_an['y_cm_dlc'] - df_an['y_cm_topscan'])**2
        )
        info_dlc = arquivos_dlc.get(BODYPART_LEGENDA, {}).get(rato_id)
        rois     = rois_definidas.get(rato_id)
        if not info_dlc or not rois:
            continue
        df_comp = preparar_dados_dlc_para_video(
            os.path.join(DIRETORIO_DLC_TXT, info_dlc), FPS_VIDEO, definicoes_objetos=rois)
        if df_comp is None:
            continue
        df_final = pd.merge(df_an, df_comp[['FrameNum', 'exploration_type']], on='FrameNum')
        if df_final.empty:
            continue

        # Boxplot por animal
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df_final, x='exploration_type', y='distancia_cm',
                    palette=['#E0E0E0', '#9E9E9E', '#616161'])
        sns.swarmplot(data=df_final, x='exploration_type', y='distancia_cm', color='black', alpha=0.6, size=4)
        plt.title(f'Erro por Estado Comportamental — {rato_id}')
        plt.xlabel('Estado (DLC)'); plt.ylabel('Erro (cm)')
        plt.tight_layout(); plt.show()

        for cond, err in df_final.groupby('exploration_type')['distancia_cm'].mean().items():
            dados_anova.append({'sujeito': rato_id, 'condicao': cond, 'erro_medio_cm': err})

    if dados_anova:
        df_av = pd.DataFrame(dados_anova)
        print("\n--- ANOVA de Medidas Repetidas ---")
        try:
            aov = pg.rm_anova(data=df_av, dv='erro_medio_cm', within='condicao', subject='sujeito', detailed=True)
            print(aov.round(4))
            p_val = aov.loc[0, 'p-GG-corr'] if aov.loc[0, 'p-spher'] < 0.05 else aov.loc[0, 'p-unc']
            if p_val < 0.05:
                posthoc = pg.pairwise_tests(data=df_av, dv='erro_medio_cm',
                                             within='condicao', subject='sujeito', padjust='bonf')
                print("\nPost-hoc (Bonferroni):")
                print(posthoc.round(4))
        except Exception as e:
            print(f"ANOVA falhou: {e}")

        # Barplot agregado
        mapa_pt = {'chao': 'Chão', 'objeto_Familiar': 'Objeto Familiar', 'objeto_Novo': 'Objeto Novo'}
        df_av['condicao_pt'] = df_av['condicao'].map(mapa_pt)
        ordem = [v for v in ['Chão', 'Objeto Familiar', 'Objeto Novo'] if v in df_av['condicao_pt'].values]
        plt.figure(figsize=(10, 7))
        sns.barplot(data=df_av, x='condicao_pt', y='erro_medio_cm', order=ordem,
                    palette=['#E0E0E0', '#9E9E9E', '#616161'], errorbar='se',
                    capsize=0.1, edgecolor='.1', linewidth=1.5)
        sns.swarmplot(data=df_av, x='condicao_pt', y='erro_medio_cm',
                      order=ordem, color='black', alpha=0.7, size=6)
        plt.xlabel('Estado Comportamental', fontweight='bold')
        plt.ylabel('Erro Médio de Distância (cm)', fontweight='bold')
        plt.tight_layout(); plt.show()

###ETAPA 4.4: Índice de Discriminação + Shapiro + Teste t

In [ ]:
print("\n=== 4.4 Índice de Discriminação ===")
    di_ts_list, di_dlc_list, ratos_di = [], [], []
    for rato_id, tempos in tempos_salvos.items():
        tt_ts  = tempos["topscan_novo"] + tempos["topscan_familiar"]
        tt_dlc = tempos["dlc_novo"]     + tempos["dlc_familiar"]
        if tt_ts > 0 and tt_dlc > 0:
            di_ts_list.append((tempos["topscan_novo"] - tempos["topscan_familiar"]) / tt_ts)
            di_dlc_list.append((tempos["dlc_novo"]    - tempos["dlc_familiar"])     / tt_dlc)
            ratos_di.append(rato_id)
            print(f"  {rato_id} | DI TopScan={di_ts_list[-1]:.4f}  DI DLC={di_dlc_list[-1]:.4f}")

    if len(ratos_di) >= 3:
        diferencas = np.array(di_dlc_list) - np.array(di_ts_list)
        stat_sw, p_sw = shapiro(diferencas)
        print(f"\nShapiro-Wilk (diferenças DI): W={stat_sw:.4f}, p={p_sw:.4f}")
        t_stat, p_t = ttest_rel(di_ts_list, di_dlc_list)
        print(f"Teste t pareado: t={t_stat:.4f}, p={p_t:.4f}")

        fig, ax = plt.subplots(figsize=(8, 8), layout='constrained')
        ax.scatter(di_ts_list, di_dlc_list, c='black', s=50, alpha=0.7)
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'r--', alpha=0.75, label='Concordância Ideal (y=x)')
        ax.set_xlabel('DI TopScan', fontsize=15); ax.set_ylabel('DI DeepLabCut', fontsize=15)
        ax.legend(); ax.set_aspect('equal', adjustable='box')
        ax.grid(True, linestyle='--')
        plt.show()

###ETAPA 4.5: Bland-Altman (Tempo total de exploração)

In [ ]:
print("\n=== 4.5 Bland-Altman ===")
    t_ts_arr  = np.array([v["topscan_novo"] + v["topscan_familiar"] for v in tempos_salvos.values()])
    t_dlc_arr = np.array([v["dlc_novo"]     + v["dlc_familiar"]     for v in tempos_salvos.values()])
    validos   = (t_ts_arr > 0) & (t_dlc_arr > 0)
    t_ts_arr, t_dlc_arr = t_ts_arr[validos], t_dlc_arr[validos]

    if len(t_ts_arr) >= 2:
        media = (t_ts_arr + t_dlc_arr) / 2
        dif   = t_ts_arr - t_dlc_arr
        vies  = np.mean(dif); sd = np.std(dif, ddof=1)
        print(f"Viés: {vies:.2f} s | Limites ±1.96 SD: [{vies - 1.96*sd:.2f}, {vies + 1.96*sd:.2f}] s")

        fig, ax = plt.subplots(figsize=(10, 7), layout='constrained')
        ax.scatter(media, dif, c='black', s=50, alpha=0.7)
        ax.axhline(vies,             color='red',  linestyle='--', label=f'Viés: {vies:.2f}')
        ax.axhline(vies + 1.96 * sd, color='gray', linestyle=':',  label='±1.96 SD')
        ax.axhline(vies - 1.96 * sd, color='gray', linestyle=':')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_xlabel('Tempo Médio de Exploração (s)')
        ax.set_ylabel('Diferença (TopScan − DLC) (s)')
        ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
        plt.show()

###ETAPA 4.6: Cross-correlação temporal (Grupo)

In [ ]:
print("\n=== 4.6 Cross-Correlação Temporal ===")
    JANELA = 100
    lags_grupo, curvas = [], []

    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None or len(df_an) < 20:
            continue
        sig_dlc = (df_an['x_cm_dlc']     - df_an['x_cm_dlc'].mean()).values
        sig_ts  = (df_an['x_cm_topscan'] - df_an['x_cm_topscan'].mean()).values
        corr    = correlate(sig_dlc * hann(len(sig_dlc)), sig_ts * hann(len(sig_ts)), mode='full')
        lags    = np.arange(-len(sig_ts) + 1, len(sig_ts))
        lag_ot  = int(lags[np.argmax(corr)])
        lags_grupo.append({'Rato_ID': rato_id, 'Lag_Otimo': lag_ot})

        idx0 = np.where(lags == 0)[0][0]
        ini, fim = idx0 - JANELA, idx0 + JANELA
        if ini >= 0 and fim < len(corr):
            recorte = corr[ini:fim]
            curvas.append(recorte / np.max(np.abs(recorte)))

    if lags_grupo:
        df_lags   = pd.DataFrame(lags_grupo)
        media_lag = df_lags['Lag_Otimo'].mean()
        std_lag   = df_lags['Lag_Otimo'].std()
        print(f"Lag médio do grupo: {media_lag:.2f} ± {std_lag:.2f} frames")

        if curvas:
            matriz     = np.array(curvas)
            media_curv = matriz.mean(axis=0)
            sem_curv   = matriz.std(axis=0) / np.sqrt(len(curvas))
            eixo_x     = np.arange(-JANELA, JANELA)

            plt.figure(figsize=(12, 7))
            plt.plot(eixo_x, media_curv, color='#1f77b4', linewidth=2, label='Cross-Correlação Média')
            plt.fill_between(eixo_x, media_curv - sem_curv, media_curv + sem_curv,
                             color='#1f77b4', alpha=0.2, label='EPM')
            plt.axvline(0,         color='red',   linestyle='--', linewidth=2, label='Lag Zero')
            plt.axvline(media_lag, color='green', linestyle=':',  linewidth=2,
                        label=f'Lag do Grupo ({media_lag:.1f} fr)')
            plt.xlabel('Defasagem (Frames)'); plt.ylabel('Correlação Normalizada')
            plt.legend(); plt.grid(True, linestyle='--', alpha=0.7)
            plt.tight_layout(); plt.show()

        plt.figure(figsize=(7, 6))
        sns.boxplot(y=df_lags['Lag_Otimo'], color='lightgray', width=0.4, showfliers=False)
        sns.swarmplot(y=df_lags['Lag_Otimo'], color='black', size=8, alpha=0.7)
        plt.axhline(0, color='red', linestyle='--', linewidth=2, label='Sincronização Perfeita')
        plt.ylabel('Lag Ótimo (Frames)'); plt.xticks([])
        plt.tight_layout(); plt.show()

###ETAPA 4.7: Análise espacial: Centro vs. Periferia

In [ ]:
print("\n=== 4.7 Erro Centro vs. Periferia ===")
    MARGEM = 0.20
    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue
        df_an['distancia_cm'] = np.sqrt(
            (df_an['x_cm_dlc'] - df_an['x_cm_topscan'])**2 +
            (df_an['y_cm_dlc'] - df_an['y_cm_topscan'])**2
        )
        centro = (
            df_an['x_cm_dlc'].between(ARENA_LARGURA_CM * MARGEM, ARENA_LARGURA_CM * (1 - MARGEM)) &
            df_an['y_cm_dlc'].between(ARENA_ALTURA_CM  * MARGEM, ARENA_ALTURA_CM  * (1 - MARGEM))
        )
        df_an['zona'] = np.where(centro, 'Centro', 'Periferia')
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df_an, x='zona', y='distancia_cm', palette='plasma')
        plt.title(f'Erro por Zona — {rato_id}')
        plt.xlabel('Zona'); plt.ylabel('Erro (cm)')
        plt.grid(True, linestyle='--', alpha=0.6, axis='y')
        plt.tight_layout(); plt.show()

###ETAPA 4.8: Perfis de velocidade (KDE + K-S)

In [ ]:
print("\n=== 4.8 Perfis de Velocidade ===")
    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue
        for sis, cx, cy, ct in [('dlc', 'x_cm_dlc', 'y_cm_dlc', 'time_dlc'),
                                  ('topscan', 'x_cm_topscan', 'y_cm_topscan', 'time_topscan')]:
            df_an[f'vel_{sis}'] = (np.sqrt(df_an[cx].diff()**2 + df_an[cy].diff()**2) /
                                   df_an[ct].diff())
        df_an = df_an.replace([np.inf, -np.inf], np.nan).dropna(subset=['vel_dlc', 'vel_topscan'])
        if df_an.empty:
            continue
        ks, pv = ks_2samp(df_an['vel_dlc'], df_an['vel_topscan'])
        print(f"  {rato_id} | K-S D={ks:.3f}, p={pv:.3g}")

        lim_x = df_an['vel_dlc'].quantile(0.995)
        plt.figure(figsize=(10, 6))
        sns.kdeplot(df_an['vel_dlc'],     label='DLC',     color='purple', fill=True, alpha=0.3)
        sns.kdeplot(df_an['vel_topscan'], label='TopScan', color='green',  fill=True, alpha=0.3)
        plt.xlabel('Velocidade (cm/s)'); plt.ylabel('Densidade')
        plt.xlim(0, lim_x); plt.legend(); plt.grid(True, linestyle='--')
        plt.title(f'Distribuição de Velocidade — {rato_id}')
        plt.tight_layout(); plt.show()

###ETAPA 4.9: Dashboard: trajetória + erro × tempo + heatmap + erro × velocidade

In [ ]:
print("\n=== 4.9 Dashboard de Validação ===")
    NUM_BINS_DASH = 50
    for rato_id in ratos_comuns:
        df_an = _sincronizar_dados_cm(rato_id, resultados_cm)
        if df_an is None:
            continue
        df_an['distancia_cm'] = np.sqrt(
            (df_an['x_cm_dlc'] - df_an['x_cm_topscan'])**2 +
            (df_an['y_cm_dlc'] - df_an['y_cm_topscan'])**2
        )
        df_an['vel_cm_s'] = (np.sqrt(df_an['x_cm_dlc'].diff()**2 + df_an['y_cm_dlc'].diff()**2) /
                             df_an['time_dlc'].diff())
        df_an = df_an.replace([np.inf, -np.inf], np.nan).dropna(subset=['vel_cm_s', 'distancia_cm'])

        rmse       = np.sqrt(np.mean(df_an['distancia_cm']**2))
        corr_vel_e, _ = pearsonr(df_an['vel_cm_s'], df_an['distancia_cm'])

        fig, axes = plt.subplots(2, 2, figsize=(18, 16), layout='constrained')
        fig.suptitle(f'Dashboard de Validação — {rato_id}\n'
                     f'RMSE={rmse:.2f} cm  |  Corr(Erro, Vel)={corr_vel_e:.2f}', fontsize=22)

        # Painel 1: trajetórias
        ax = axes[0, 0]
        ax.plot(df_an['x_cm_dlc'], df_an['y_cm_dlc'],     color='purple', alpha=0.6, label='DLC')
        ax.plot(df_an['x_cm_topscan'], df_an['y_cm_topscan'], color='green', alpha=0.6, linestyle='--', label='TopScan')
        ax.set_xlim(0, ARENA_LARGURA_CM); ax.set_ylim(0, ARENA_ALTURA_CM)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title('Alinhamento de Trajetórias', fontsize=19, fontweight='bold')
        ax.set_xlabel('X (cm)', fontsize=15); ax.set_ylabel('Y (cm)', fontsize=15)
        ax.legend(fontsize=13); ax.grid(True, linestyle=':')

        # Painel 2: erro × tempo
        ax = axes[0, 1]
        ax.plot(df_an['FrameNum'], df_an['distancia_cm'], color='blue', alpha=0.7)
        media_e = df_an['distancia_cm'].mean()
        ax.axhline(media_e, color='orange', linestyle='--', label=f'Média ({media_e:.2f} cm)')
        ax.set_title('Erro Posicional × Tempo', fontsize=19, fontweight='bold')
        ax.set_xlabel('Frame', fontsize=15); ax.set_ylabel('Erro (cm)', fontsize=15)
        ax.legend(fontsize=13); ax.grid(True, linestyle=':')

        # Painel 3: heatmap
        ax = axes[1, 0]
        bins_x = np.linspace(0, ARENA_LARGURA_CM, NUM_BINS_DASH + 1)
        bins_y = np.linspace(0, ARENA_ALTURA_CM,  NUM_BINS_DASH + 1)
        hm_s, _, _ = np.histogram2d(df_an['x_cm_dlc'], df_an['y_cm_dlc'],
                                     bins=[bins_x, bins_y], weights=df_an['distancia_cm'])
        hm_c, _, _ = np.histogram2d(df_an['x_cm_dlc'], df_an['y_cm_dlc'], bins=[bins_x, bins_y])
        hm_avg = np.divide(hm_s, hm_c, out=np.zeros_like(hm_s), where=hm_c != 0)
        im = ax.imshow(hm_avg.T, origin='lower', cmap='jet',
                       extent=[0, ARENA_LARGURA_CM, 0, ARENA_ALTURA_CM])
        cbar = fig.colorbar(im, ax=ax); cbar.set_label('Erro Médio (cm)', fontsize=12)
        ax.set_title('Mapa de Calor do Erro', fontsize=19, fontweight='bold')
        ax.set_xlabel('X (cm)', fontsize=15); ax.set_ylabel('Y (cm)', fontsize=15)

        # Painel 4: erro × velocidade
        ax = axes[1, 1]
        ax.scatter(df_an['vel_cm_s'], df_an['distancia_cm'], alpha=0.2, s=10)
        m, b = np.polyfit(df_an['vel_cm_s'], df_an['distancia_cm'], 1)
        x_l = np.linspace(df_an['vel_cm_s'].min(), df_an['vel_cm_s'].max(), 100)
        ax.plot(x_l, m * x_l + b, color='red', linewidth=2)
        r_v, _ = pearsonr(df_an['vel_cm_s'], df_an['distancia_cm'])
        ax.text(0.05, 0.9, f'r = {r_v:.2f}', transform=ax.transAxes,
                fontweight='bold', color='red', fontsize=12)
        ax.set_title('Erro × Velocidade', fontsize=19, fontweight='bold')
        ax.set_xlabel('Velocidade (cm/s)', fontsize=15); ax.set_ylabel('Erro (cm)', fontsize=15)
        ax.grid(True, linestyle=':')

        # layout='constrained' gerencia o espaçamento automaticamente
        plt.show()

#PONTO DE ENTRADA — executa a pipeline completa

In [ ]:
def executar_pipeline():
    # Etapa 0
    ratos_comuns, arquivos_topscan, arquivos_dlc = carregar_dados()

    # Etapa 1
    resultados_cm = processar_trajetorias(ratos_comuns, arquivos_topscan, arquivos_dlc)

    # Etapa 1.2 e 1.3
    plotar_trajetorias_comparativas(resultados_cm)
    plotar_trajetorias_brutas(ratos_comuns, arquivos_topscan, arquivos_dlc)

    # Etapa 2
    rois_definidas = definir_rois(ratos_comuns)
    verificar_rois(rois_definidas, arquivos_topscan)
    gerar_videos(ratos_comuns, arquivos_topscan, arquivos_dlc, resultados_cm, rois_definidas)

    # Etapa 3
    tempos_salvos = calcular_exploracao_comportamental(
        ratos_comuns, arquivos_topscan, arquivos_dlc, rois_definidas)

    # Etapa 4
    executar_estatistica(ratos_comuns, resultados_cm, arquivos_dlc, rois_definidas, tempos_salvos)


if __name__ == '__main__':
    executar_pipeline()